[Guide Index](README.md)


# 03.03 Preparing, extending and checking queries

What an application builds on top of `query()`: the triples an answer depends on, a query prepared once for re-use, custom Python functions and aggregates, and checking a query before it is run. See [SPARQL](03.01-sparql.ipynb) for running a query and reading its result.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph import StarLayerGraph

EX = "http://example.org/"
RDF = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"

## 03.03.1 What an answer depends on

`graph.relevant_subgraph(query)` returns the triples of the graph that the answer to a query depends on, as a new graph. Every way the answer can be derived is kept. It takes a SELECT, ASK or CONSTRUCT query.

On a dataset it returns a dataset of quads, each in the graph it was read from. A `GRAPH` block keeps the quads of the graph it names. A pattern outside a `GRAPH` block reads the default graph (the union of all graphs for a `default_union` dataset), and keeps a quad in every graph that holds it.

In [2]:
from starlayer.graph import StarLayerDataset

evidence = StarLayerGraph()
evidence.parse(data='''@prefix ex: <http://example.org/> .
ex:teamA ex:name "ann" ; ex:size 2 .
ex:teamB ex:name "bo" ; ex:size 9 .''', format="turtle")
used = evidence.relevant_subgraph("SELECT ?t WHERE { ?t ex:name 'bo' ; ex:size ?n FILTER(?n > 5) }")
print(len(evidence), "triples in the graph;", len(used), "that the answer depends on:")
for s, p, o in sorted(used.triples(), key=str):
    print(" ", s.value, p.value, o.value)

# a dataset gives a dataset of quads
dataset = StarLayerDataset()
dataset.parse(data='''@prefix ex: <http://example.org/> .
ex:teamA ex:size 2 .
ex:teamB ex:size 9 .
ex:archive { ex:teamB ex:size 9 . ex:teamC ex:size 1 }''', format="trig")
used = dataset.relevant_subgraph("SELECT ?t WHERE { ?t ex:size ?n FILTER(?n > 5) }")
print()
print(len(list(dataset.quads())), "quads in the dataset;", len(list(used.quads())), "that the answer depends on:")
for q in sorted(used.quads(), key=str):
    print(" ", q.subject.value, q.predicate.value, q.object.value, "| graph:", q.graph_name)

# a GRAPH block keeps the quads of the graph it names
print("\nDataset exampe using a GRAPH block:")
used = dataset.relevant_subgraph("SELECT ?t WHERE { GRAPH ex:archive { ?t ex:size ?n FILTER(?n > 5) } }")

for q in sorted(used.quads(), key=str):
    print(" ", q.subject.value, q.predicate.value, q.object.value, "| graph:", q.graph_name)


4 triples in the graph; 2 that the answer depends on:
  http://example.org/teamB http://example.org/name bo
  http://example.org/teamB http://example.org/size 9

4 quads in the dataset; 2 that the answer depends on:
  http://example.org/teamB http://example.org/size 9 | graph: DEFAULT
  http://example.org/teamB http://example.org/size 9 | graph: <http://example.org/archive>

Dataset exampe using a GRAPH block:
  http://example.org/teamB http://example.org/size 9 | graph: <http://example.org/archive>


## 03.03.2 Preparing a query for re-use

`StarQuery(text, ...)` from `starlayer.sparql` allows for a query object to be created and then run against a graphs with `run(graph)`. It is for a query that an application creates to be run against many graphs. A syntax error is raised when the query is created.

The options of `query()` are bound once when the query is made. The ones used most often:

- `initNs`: the prefixes the query text uses, so the text needs no `PREFIX` lines.
- `initBindings`: variables that are bound before the query runs.
- `entailment`: the entailment regime to query under (see the [inferencing guide](02.06-graphs-inferencing.ipynb)).
- `timeout`: the seconds the OWL-Direct reasoner may take, which applies to that regime only.

`base`, `recompute` and `engine` are also accepted, as for `query()`. Any option can be changed for a single run, such as `run(graph, entailment=None)` to run without entailment. The query itself keeps its bound value.

In [3]:
from starlayer.errors import FormatError
from starlayer.graph import ENTAILMENT, NamedNode
from starlayer.sparql import StarQuery

# Two graphs to run the queries against.
# people: ann is a Member.
people = StarLayerGraph()
people.parse(data="@prefix ex: <http://example.org/> . ex:ann a ex:Member .", format="turtle")
# staff: bo is a Chair, cy and di are Clerks, and a Chair is a kind of Member.
staff = StarLayerGraph()
staff.parse(data="""@prefix ex: <http://example.org/> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Chair rdfs:subClassOf ex:Member . ex:bo a ex:Chair . ex:cy a ex:Clerk . ex:di a ex:Clerk .""", format="turtle")

def names(result):
    """The ?x values of a result in order, to keep the prints short."""
    return sorted(row.x.value for row in result)

# initNs: the prefix is bound once when the query is made, so the text needs no PREFIX line.
# entailment: the regime is bound once as well, so every run of this query uses RDFS.
membersQuery = StarQuery("SELECT ?x WHERE { ?x a ex:Member }", initNs={"ex": EX}, entailment=ENTAILMENT.RDFS)

# One query object runs against both graphs. Under RDFS a Chair is a Member, so bo is found.
result = membersQuery.run(people)
print("members of people:", names(result))
result = membersQuery.run(staff)
print("members of staff:", names(result))

# A single run can set entailment to None. bo is no longer found: nothing else says a Chair is a Member.
result = membersQuery.run(staff, entailment=None)
print("members of staff, entailment=None:", names(result))

# initBindings: variables that are bound before the query runs. The values are terms, not prefixed names.
# Given in run(), they apply to that run only, so one query text can be asked about each class in turn.
classQuery = StarQuery("SELECT ?x WHERE { ?x a ?class }")
result = classQuery.run(staff, initBindings={"class": NamedNode(EX + "Chair")})
print("chairs of staff:", names(result))
result = classQuery.run(staff, initBindings={"class": NamedNode(EX + "Clerk")})
print("clerks of staff:", names(result))

# A syntax error is raised when the query is made, not when it is first run.
try:
    StarQuery("SELECT ?x WHERE { ?x a }")
except FormatError as error:
    print("rejected when made:", str(error).splitlines()[0])


members of people: ['http://example.org/ann']
members of staff: ['http://example.org/bo']
members of staff, entailment=None: []
chairs of staff: ['http://example.org/bo']
clerks of staff: ['http://example.org/cy', 'http://example.org/di']
rejected when made: SPARQL syntax error at line 1, column 24: expected an RDF term or variable (found '}')


## 03.03.3 Custom functions and aggregates

A query can call functions written in Python. `query()` and `update()` take them for one call, in two forms:

- `custom_functions`: a function works on one row. It takes the row's value, a pyoxigraph term, and returns a term, or `None` to leave the variable unbound. It can be used wherever SPARQL allows an expression: in the `SELECT`, or in a `BIND` or `FILTER` in the `WHERE` clause.
- `custom_aggregate_functions`: an aggregate works on a group of rows and gives one result for the group, so it is a class, and Oxigraph expects it in a fixed form. It is called with no arguments, once for each group, and it must return an object with two methods of these names: `accumulate(term)`, called for each value in the group, and `finish()`, called at the end, which returns the result as a term. An aggregate is used with `GROUP BY`.

Each maps a function IRI (a `NamedNode` or its text) to the Python function or class, and the query calls it by that IRI, as in `<urn:fn:shout>(?n)`. A query that calls an IRI that was not passed raises an `ExecutionError` (kind `"query"`) carrying Oxigraph's "custom function ... is not supported".

In [4]:
from starlayer.errors import ExecutionError
from starlayer.graph import Literal

# A function: called once for each row, with that row's value.
def shout(term):
    return Literal(term.value.upper())

# An aggregate has to be in the form Oxigraph expects. custom_aggregate_functions maps the IRI to something Oxigraph can call
# with no arguments (here the class), once for each group. What it returns must be an object with exactly two methods:
#   accumulate(self, term)  is called for each value in the group, with the value as a pyoxigraph term. It returns nothing.
#   finish(self)            is called once, after the last value, and returns the result for the group as a term.
# Any other method name, or an __init__ that needs arguments, will not work. Whatever the aggregate keeps between the calls
# has to be kept in the object. A new object is made for each group, here each team.
class Join:
    def __init__(self):
        self.parts = []  # no arguments. Called when the group starts: an empty list to collect the names in

    def accumulate(self, term):
        self.parts.append(term.value)  # one value of the group: keep it (the values are not in a set order)

    def finish(self):
        return Literal(", ".join(sorted(self.parts)))  # all the values have been seen: return the one result, a term

g_fn = StarLayerGraph()
g_fn.parse(data='''@prefix ex: <http://example.org/> .
ex:teamA ex:name "ann", "al" .
ex:teamB ex:name "bo" .''', format="turtle")

# shout is called for each row, so teamA gives two rows.
rows = g_fn.query('SELECT ?team (<urn:fn:shout>(?n) AS ?loud) WHERE { ?team ex:name ?n }', custom_functions={"urn:fn:shout": shout})
print(sorted((r.team.value, r.loud.value) for r in rows))

# The same function in the WHERE clause. BIND gives the result of the function a variable for each row.
rows = g_fn.query('SELECT ?n ?loud WHERE { ?team ex:name ?n BIND(<urn:fn:shout>(?n) AS ?loud) }', custom_functions={"urn:fn:shout": shout})
print(sorted((r.n.value, r.loud.value) for r in rows))

# A FILTER can compare the result of the function: only the rows where shout(?n) is "ANN" are kept.
rows = g_fn.query('SELECT ?n WHERE { ?team ex:name ?n FILTER(<urn:fn:shout>(?n) = "ANN") }', custom_functions={"urn:fn:shout": shout})
print(sorted(r.n.value for r in rows))

# GROUP BY ?team makes one group for each team. Join collects the names of the group and gives one string for it.
rows = g_fn.query('SELECT ?team (<urn:fn:join>(?n) AS ?names) WHERE { ?team ex:name ?n } GROUP BY ?team',
                  custom_aggregate_functions={"urn:fn:join": Join})
print(sorted((r.team.value, r.names.value) for r in rows))

# a function that was not passed
try:
    list(g_fn.query('SELECT (<urn:fn:shout>(?n) AS ?loud) WHERE { ?team ex:name ?n }'))
except ExecutionError as error:
    print("not passed:", error)


[('http://example.org/teamA', 'AL'), ('http://example.org/teamA', 'ANN'), ('http://example.org/teamB', 'BO')]
[('al', 'AL'), ('ann', 'ANN'), ('bo', 'BO')]
['ann']
[('http://example.org/teamA', 'al, ann'), ('http://example.org/teamB', 'bo')]
not passed: The custom function <urn:fn:shout> is not supported


## 03.03.4 Checking a query

`check_query(text)` checks a SPARQL 1.2 query or update before it is run, so errors or inconsistencies can be identified before the query or update is executed. It checks the grammar and the `VERSION` line (see [the `VERSION` directive](03.02-sparql-12-queries.ipynb)), warns about likely mistakes, such as a variable that is never bound, and runs a query against an empty Oxigraph, which finds what Oxigraph itself rejects. It never runs the query on the data.

It returns a `QueryCheck`: `ok` is false when there is an error, and `problems` lists everything found, each with a kind, a severity (`"error"` or `"warning"`) and a message.

A query may use prefixes that are defined by the graph that will be used with the query. Without `prefixes=`, an undeclared prefix is returned as a warning. To check against a graph's prefixes, use the graph's own method, `graph.check_query(text)`, or pass them: `check_query(text, prefixes=graph.prefixes)`.

A query that calls custom functions needs them passed, as for `query()`: `check_query(text, custom_functions=...)`, because the empty Oxigraph does not know them otherwise (section 03.03.3).

In [5]:
from starlayer.sparql import check_query

def show(check):
    print(check.form, "| ok" if check.ok else "| not ok")
    for problem in check.problems:
        where = f" (line {problem.line}, column {problem.column})" if problem.line else ""
        print("  ", problem.kind, problem.severity + ":", problem.message + where)

# check_query(text): the grammar, the VERSION line, Oxigraph and likely mistakes
PREFIX = "PREFIX ex: <http://example.org/> "
show(check_query(PREFIX + "SELECT ?s WHERE { ?s ex:name ?n }"))
show(check_query(PREFIX + "SELECT ?s WHERE { ?s ex:name }"))
show(check_query('VERSION "1.1" ' + PREFIX + "SELECT ?s WHERE { ?s ex:p <<( ex:a ex:b ex:c )>> }"))
# ?z is never bound: legal SPARQL, so a warning and the text is still ok
show(check_query(PREFIX + "SELECT ?z WHERE { ?s ex:name ?n }"))

# graph.check_query(text): the prefixes of the graph the text will run against are declared
g_prefixed = StarLayerGraph(prefixes={"ex": EX})
text = "SELECT ?s WHERE { ?s ex:name ?n }"
show(check_query(text))
show(g_prefixed.check_query(text))

# custom_functions=...: the empty Oxigraph does not know a function unless it is passed
text = "SELECT ?s WHERE { ?s ?p ?o FILTER(<urn:fn:unknown>(?o)) }"
show(check_query(text))
show(check_query(text, custom_functions={"urn:fn:unknown": lambda term: term}))


SELECT | ok
None | not ok
   syntax error: expected an RDF term or variable (found '}') (line 1, column 63)
SELECT | not ok
   version error: SPARQL query declares VERSION '1.1' but uses a triple term, which '1.1' conformance excludes (SPARQL 1.2 Query sec 4.3)
SELECT | ok
   shape warning: Projected variable z is never bound anywhere in this query's own sqe:where pattern.
SELECT | ok
   prefix warning: undeclared prefix 'ex' (found 'ex:name') (line 1, column 22)
SELECT | ok
SELECT | not ok
   engine error: The custom function <urn:fn:unknown> is not supported
SELECT | ok
